![DB Academy](./Includes/images/db-academy.png)


<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">LAB, Bonus Capstone</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Capstone, Refining Retail Analytics</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Pull together everything from Lessons 02 through 11, extend a real dashboard with new datasets and visualizations, refresh its filter, republish, and bolt on a Genie Agent for self-service follow-up questions.</div>
</div>

</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select SQL Warehouse Compute</strong>
  <div style="color:#333;">

This notebook requires a <strong>SQL Warehouse</strong> for execution. Before running any cells, attach this notebook to a SQL Warehouse. Select the compute tile in the top-right corner of the notebook, then choose your SQL Warehouse from the list.
  </div>
</div>

## PREREQUISITE - ENVIRONMENT SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">
    Required Setup
  </strong>
  <div style="color:#333;">
  Before running this notebook, make sure you have already executed the <strong>Required Setup</strong>, in the <strong>00 - Course Overview</strong> notebook, to properly setup your catalog, schema, and files for this notebook.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.

In [0]:
%run ./Includes/Classroom-Setup

**The scenario:** management has asked you to enhance the **Retail Dashboard** you built in Lesson 08 with an additional dataset and a couple more visualizations. They've also asked you to enable self-service question-answering on the result, so questions that come in while you're out of office can be handled without you. You'll be back in front of stakeholders in 30 minutes. Let's go.

In this capstone you will:

1. Add the `sales_orders` table to the dashboard's data layer.
2. Add a Pie Chart of sales by product category.
3. Write two new SQL datasets that join across tables (one for *Sales by loyalty segment*, one for *Sales and orders by day*).
4. Build a Heatmap and a dual-axis Line Chart from those datasets.
5. Update the existing Filter so it covers the new datasets, then republish.
6. Auto-generate a Genie Agent from the published dashboard.

*Prerequisite: Complete **Lesson 08** (the Retail Dashboard) and **Lesson 10** (Creating Genie Agents) before starting this capstone. The dashboard you extend here is the one you built in Lesson 08; the Genie integration at the end uses techniques from Lesson 10.*

### Task 1: Add the `sales_orders` dataset

1. In the left sidebar, click **Dashboards** and open your **Retail Dashboard** from Lesson 08.
2. Switch to the **Edit Draft** view (use the context icon next to the dashboard name).
3. Click the **Data** tab.
4. Click **+ Add dataset** and add the **`sales_orders`** table from your `retail` schema.

### Task 2: Build a Pie Chart: sales by product category

Show the proportion of sales coming from each product category.

1. Switch back to the **Untitled page** canvas tab.
2. Click **Add a visualization**, drop the widget on the canvas.
3. In the **Configuration Panel**, set:
   - **Dataset:** *sales*
   - **Visualization:** *Pie*
   - **Angle:** *product_category*, apply the **COUNT** transformation
   - **Color:** *product_category*
4. Optionally rename the widget title to *Sales by Product Category*.

### Task 3: SQL dataset: *Sales by loyalty segment*

The next two charts need data spanning multiple tables. Write a SQL dataset that joins `sales` and `customers`, returning the columns needed to plot revenue by loyalty segment and product category.

The query should return three columns:

- `product_category`: from `sales`
- `loyalty_segment`: from `customers`
- `total_price`: from `sales`

Run the cell below to verify your query, then copy the working SQL into a new SQL dataset on the dashboard's **Data** tab and rename it to **Sales by loyalty segment**.

In [0]:
-- TODO: Join sales and customers on customer_id; return product_category, loyalty_segment, total_price.
SELECT
  product_category,
  loyalty_segment,
  total_price
FROM <FILL_IN>           -- the sales table
JOIN <FILL_IN>           -- the customers table
  ON <FILL_IN>;

<details>
<summary>Hint</summary>

<pre><code>SELECT
  product_category,
  loyalty_segment,
  total_price
FROM sales
JOIN customers
  ON sales.customer_id = customers.customer_id;</code></pre>

</details>

You should see one row per sale, with that sale's product category, the customer's loyalty segment, and the total price. The row count will match the row count of `sales`.

### Task 4: SQL dataset: *Sales and orders by day*

The dual-axis chart needs revenue *and* order count grouped by **day of the month**. The trick: `sales.order_date` is a `DATE`, but `sales_orders.order_datetime` is a `BIGINT` Unix timestamp. You'll aggregate each separately, then join the per-day results.

The query below has the structure already laid out. Fill in the table references and final ordering. Run it to verify before pasting into the dashboard. Rename the dataset to **Sales and orders by day**.

In [0]:
-- TODO: Aggregate revenue from sales and order count from sales_orders by day, then join.
WITH sales_data AS (
  SELECT
    date_format(order_date, 'dd') AS day,
    SUM(total_price) AS total_sales
  FROM <FILL_IN>          -- the sales table
  GROUP BY day
),
orders_data AS (
  SELECT
    CASE
      WHEN try_cast(sales_orders.order_datetime AS BIGINT) IS NOT NULL
      THEN DAY(FROM_UNIXTIME(sales_orders.order_datetime))
      ELSE NULL
    END AS day,
    COUNT(order_number) AS total_orders
  FROM <FILL_IN>          -- the sales_orders table
  GROUP BY day
)
SELECT
  cast(s.day AS INT) AS day,
  s.total_sales,
  o.total_orders
FROM sales_data s
JOIN orders_data o ON s.day = o.day
ORDER BY <FILL_IN>;       -- sort oldest day first

<details>
<summary>Hint</summary>

The two table refs are <code>sales</code> and <code>sales_orders</code>. The final sort is by <code>day</code> ascending. You can use either <code>s.day</code> or just <code>day</code> since you aliased the cast result.
<br/><br/>
<pre><code>...
FROM sales
...
FROM sales_orders
...
ORDER BY day;</code></pre>

</details>

You should see one row per day-of-month (1 through 31, more or less, depending on the data) with `total_sales` and `total_orders` aligned. If your output looks wrong, the most common cause is forgetting that `order_datetime` is a Unix timestamp, the `CASE` expression handles that conversion for you.

### Task 5: Build the Heatmap and the dual-axis Line Chart

Two visualizations, both on the canvas. No FILL_IN, both are configured in the UI.

#### 5a: Heatmap (revenue by category and loyalty segment)

1. **Add a visualization**, drop on the canvas.
2. Configure:
   - **Dataset:** *Sales by loyalty segment*
   - **Visualization:** *Heatmap*
   - **X axis:** *product_category*
   - **Y axis:** *loyalty_segment*
   - **Color:** *SUM(total_price)*
3. Resize the widget so all categories fit, the default is too small.

#### 5b: Dual-axis Line Chart (revenue and orders by day)

1. **Add a visualization**, drop on the canvas.
2. Configure:
   - **Dataset:** *Sales and orders by day*
   - **Visualization:** *Line*
   - **X axis:** *day*, Transformation: **None**
   - **Y axis:** *SUM(total_sales)*
   - **Y axis:** *SUM(total_orders)*
3. Click the kebab `(⋮)` next to **Y axis** and toggle **Enable dual axis** so each series uses its own scale.

Two readings to make from the new charts: which loyalty segment over-indexes on certain categories (Heatmap), and whether revenue and order count move together day-to-day (Dual-axis Line. They don't always).

### Task 6: Update the Filter, then republish

The Filter widget you added in Lesson 08 was bound to `sales.product_category` and `Three Month Sales.product_category`. It needs the new dataset added so it filters the Heatmap too.

1. Click the existing Filter widget on the canvas to select it.
2. In the configuration panel, click the **+** next to **Fields**.
3. Add **`Sales by loyalty segment.product_category`**.
4. Test the filter from the dropdown, picking a product category should now affect the Heatmap as well as the original visualizations.
5. Click **Publish** in the top-right to push your changes. Leave **Share data permission** as default and confirm.

<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.05em;">Warning: viewers won't see your edits until you republish</strong>
<div style="color:#333;">

Edits in the Edit Draft view are invisible to dashboard viewers until you click <strong>Publish</strong> again. The Sales-and-Orders charts you just added (and the broadened filter) won't appear in <em>View Published</em> until you republish.

</div>
</div>

### Task 7 (Final integration): Auto-generate a Genie Agent from the dashboard

One last step closes the loop on the scenario, *self-service question-answering while you're out of office*. You used this exact technique in Lesson 10's second walkthrough; here you'll do it on the published Retail Dashboard.

1. From the **Edit Draft** view, click the kebab `(⋮)` in the top-right and select **Settings and themes**.
2. In the **General** tab, toggle **Enable Genie**.
3. When prompted, choose **Auto-generate Genie agent**.
4. Click **Publish** again to push the Genie integration along with your latest visualizations.
5. Switch to **View Published**. An **Ask Genie** button appears at the bottom of the dashboard.
6. Click **Ask Genie** and try one stakeholder-style question:

   > *Which day of the month had the highest total revenue, and how many orders happened that day?*

Genie should answer using the *Sales and orders by day* dataset you just built, proof that adding datasets to a dashboard automatically widens what its Genie Agent can answer.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.05em;">Information: auto-generated Agents are starting points</strong>
<div style="color:#333;">

As you saw in Lesson 10, an auto-generated Agent inherits the dashboard's datasets but has no description, common questions, instructions, or example queries. Before handing this Agent to stakeholders, you'd revisit <strong>Configure → About</strong>, <strong>Instructions</strong>, and <strong>Example SQL queries</strong> to bring it to production quality. Out of scope for this 30-minute capstone, but a useful next step.

</div>
</div>


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">Nice work! You just completed the capstone:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Extended an existing dashboard with a new whole-table dataset (<code>sales_orders</code>) and two SQL datasets that span multiple tables.</li>
      <li>Wrote a clean <code>JOIN</code> for revenue by loyalty segment, and a CTE-based aggregation that bridges a <code>DATE</code> column and a Unix-timestamp column on day-of-month.</li>
      <li>Added a Pie Chart, a Heatmap, and a dual-axis Line Chart, three visualization patterns you'll reuse on real dashboards.</li>
      <li>Updated an existing Filter so it spans the new dataset, then republished the dashboard.</li>
      <li>Auto-generated a Genie Agent directly from the dashboard, closing the loop on the scenario's <em>self-service while you're out of office</em> requirement.</li>
    </ul>
    <div style="margin-top: 12px;">You've now exercised every capability in this course end-to-end on a single business scenario, discovery, querying, Delta Lake, dashboards, and Genie. Wrap up with the <strong>Summary and Next Steps</strong> notebook for related courses and the <strong>Data Analysis Associate</strong> certification path.</div>
  </div>
</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>